# Lecture 8 — CNN Architectures

**Course:** UMich EECS 498-007 / 598-005 (Justin Johnson), Fall 2019, Lecture 8.  
**Merged with:** Stanford CS231n 2026 [Lecture 6: *Training CNNs and CNN Architectures*](https://cs231n.stanford.edu/slides/2026/lecture_6.pdf) (the architecture part; its training topics, initialization, augmentation, transfer learning, and hyperparameter search, are in Lectures 10–11 of this series).

> **About the sources.** UMich-specific content is based on knowledge of Justin Johnson's lecture; the CS231n 2026 slides were read in full. **[CS231n 2026]** marks content only there; **🔎** marks material beyond both. Every parameter and compute number here is **computed** by code in this notebook, and checked against published values where those are known exactly.

### How to read this notebook
* Each **code cell does one thing**; explanations come before it and **"What to notice"** after important results.
* We first build a small **cost calculator** from per-layer formulas, checking each formula **by hand** on AlexNet's first layer. Then we use it to compare whole architectures.
* **✍️ Try it** boxes suggest experiments. Run cells **in order**.

| § | Topic | What you'll compute |
|---|---|---|
| 1 | A cost calculator for CNN layers | Output size, memory, parameters, multiply-adds |
| 2 | ImageNet winners; **AlexNet** | Where memory, parameters, and compute live |
| 3 | ZFNet | The cost of "just make it bigger" |
| 4 | **VGG** | 138,357,544 parameters exactly; why 3×3; equal compute per stage |
| 5 | **GoogLeNet** | Stem, Inception module with 1×1 bottlenecks, global average pooling |
| 6 | **ResNet** | The degradation problem, reproduced; exact ResNet-18…152 counts; bottlenecks |
| 7 | Improving ResNets | Pre-activation; ResNeXt's equal-cost condition; SENet |
| 8 | Efficient networks; NAS; 🔎 after 2019 | Depthwise-separable savings |
| 9 | Comparing architectures; which to use? | Accuracy vs. compute |
| 10 | PyTorch model zoo | (Colab) |

### Setup

In [ ]:
import numpy as np                          # arrays and math
import matplotlib.pyplot as plt             # plots
from sklearn.datasets import load_digits    # digits, for the degradation experiment in Sec. 6
import math, time

np.set_printoptions(precision=4, suppress=True)

---
## 1. A cost calculator for CNN layers

For each layer we want the four numbers the UMich lecture tabulates. **The formulas:**

| Quantity | Convolution ($C_{\text{out}}$ filters, $K\times K$, stride $S$, pad $P$, groups $G$) | Fully connected ($D_{\text{in}}\to D_{\text{out}}$) |
|---|---|---|
| Output size | $H' = \lfloor(H - K + 2P)/S\rfloor + 1$ | $D_{\text{out}}$ |
| Memory (output activations) | $C_{\text{out}}H'W'\times4$ bytes (float32) | $D_{\text{out}}\times4$ bytes |
| Parameters | $C_{\text{out}}\cdot(C_{\text{in}}/G)\cdot K^2$ (+ $C_{\text{out}}$ biases) | $D_{\text{in}}D_{\text{out}} + D_{\text{out}}$ |
| Multiply-adds | (output elements) × (work per element) $= C_{\text{out}}H'W'\cdot(C_{\text{in}}/G)K^2$ | $D_{\text{in}}D_{\text{out}}$ |

Pooling layers change the size but have **no parameters** (and negligible compute). (One multiply-add is often counted as 2 FLOPs; papers vary, so always check which convention a number uses.)

**By hand first: AlexNet's first layer.** Input 3×227×227; 64 filters of 11×11, stride 4, pad 2:

In [ ]:
C_in, H_in, C_out, K, S, P = 3, 227, 64, 11, 4, 2
H_out = (H_in - K + 2 * P) // S + 1
outputs = C_out * H_out * H_out
print(f"output size: ({H_in} - {K} + 2*{P}) // {S} + 1 = {H_out}  ->  output {C_out} x {H_out} x {H_out}")
print(f"memory: {outputs:,} numbers x 4 bytes = {outputs * 4 / 1024:,.0f} KB")
print(f"parameters: {C_out} x {C_in} x {K} x {K} + {C_out} biases = {C_out * C_in * K * K + C_out:,}")
print(f"multiply-adds: {outputs:,} outputs x ({C_in}*{K}*{K} = {C_in * K * K}) each = {outputs * C_in * K * K / 1e6:.1f} million")

**Now as functions, one per layer type.** Each takes the current shape `(C, H, W)` plus the layer's settings, and returns the new shape, parameters, and multiply-adds:

In [ ]:
def conv_cost(shape, C_out, K, S, P, groups=1, bias=True):
    # A convolution layer. Returns (new shape, parameters, multiply-adds).
    C, H, W = shape
    H2, W2 = (H - K + 2 * P) // S + 1, (W - K + 2 * P) // S + 1
    params = C_out * (C // groups) * K * K + (C_out if bias else 0)
    macs = C_out * H2 * W2 * (C // groups) * K * K
    return (C_out, H2, W2), params, macs

def pool_cost(shape, K, S, P=0):
    # Max or average pooling: changes the size, no parameters, (negligible) compute counted as 0.
    C, H, W = shape
    return (C, (H - K + 2 * P) // S + 1, (W - K + 2 * P) // S + 1), 0, 0

def fc_cost(shape, D_out):
    # A fully connected layer on the flattened input.
    D_in = int(np.prod(shape))
    return (D_out, 1, 1), D_in * D_out + D_out, D_in * D_out

print("conv_cost on AlexNet's first layer:", conv_cost((3, 227, 227), 64, 11, 4, 2))

**The calculator** runs a list of layer specs through these functions and prints a table. Each spec is a tuple: `("conv", C_out, K, S, P[, groups, bias])`, `("pool", K, S, P)`, `("gap",)` (global average pooling), `("flatten",)`, or `("fc", D_out)`.

In [ ]:
def analyze(input_shape, layers, verbose=True, name=""):
    # Cost of a whole network. Returns (per-layer rows, total memory in KB, total params, total multiply-adds).
    shape, rows = input_shape, []
    for spec in layers:
        kind = spec[0]
        if kind == "conv":
            _, C_out, K, S, P, *rest = spec
            groups = rest[0] if rest else 1; bias = rest[1] if len(rest) > 1 else True
            shape, p, f = conv_cost(shape, C_out, K, S, P, groups, bias)
            label = f"conv {K}x{K}, {C_out}, s{S}" + (f", g{groups}" if groups > 1 else "")
        elif kind == "pool":
            shape, p, f = pool_cost(shape, *spec[1:]); label = f"pool {spec[1]}x{spec[1]}, s{spec[2]}"
        elif kind == "gap":
            shape, p, f = (shape[0], 1, 1), 0, 0; label = "global avg pool"
        elif kind == "flatten":
            shape, p, f = (int(np.prod(shape)), 1, 1), 0, 0; label = "flatten"
        elif kind == "fc":
            shape, p, f = fc_cost(shape, spec[1]); label = f"fc {spec[1]}"
        memory_kb = int(np.prod(shape)) * 4 / 1024
        rows.append((label, shape, memory_kb, p, f))
    total_mem, total_p, total_f = (sum(r[i] for r in rows) for i in (2, 3, 4))
    if verbose:
        print(f"{name}\n{'layer':22s} {'output':>15s} {'memory (KB)':>12s} {'params (k)':>11s} {'MACs (M)':>10s}")
        for label, shp, mem, p, f in rows:
            print(f"{label:22s} {str(shp):>15s} {mem:12,.0f} {p / 1e3:11,.0f} {f / 1e6:10,.0f}")
        print(f"{'TOTAL':22s} {'':>15s} {total_mem / 1024:9,.1f} MB {total_p / 1e6:9,.1f}M {total_f / 1e9:8,.2f}G")
    return rows, total_mem, total_p, total_f

---
## 2. ImageNet winners, and AlexNet

| Year | Winner | Depth | Top-5 error |
|---|---|---|---|
| 2010 | Lin et al. (NEC-UIUC) | shallow | 28.2% |
| 2011 | Sánchez & Perronnin (XRCE) | shallow | 25.8% |
| 2012 | Krizhevsky et al. (**AlexNet**) | 8 layers | 16.4% |
| 2013 | Zeiler & Fergus (**ZFNet**) | 8 layers | 11.7% |
| 2014 | Simonyan & Zisserman (**VGG**), 2nd place | 19 layers | 7.3% |
| 2014 | Szegedy et al. (**GoogLeNet**) | 22 layers | 6.7% |
| 2015 | He et al. (**ResNet**) | 152 layers | 3.57% |
| 2016 | Shao et al. (Trimps-Soushen, ensemble) | 152 layers | 2.99% |
| 2017 | Hu et al. (**SENet**) | 152 layers | 2.25% |
| | Human (Russakovsky et al.) | | 5.1% |

**"The revolution of depth"** **[CS231n 2026]**: from 8 layers to 152 in three years.

### 2.1 AlexNet (Krizhevsky, Sutskever & Hinton, 2012)
227×227 inputs; **5 convolutional layers**, max pooling, **3 fully connected layers**, **ReLU**. It also used **local response normalization** (no longer used), and was trained on **two GTX 580 GPUs** with 3 GB of memory each, with the model split between them. It's among the most cited papers in all of science. The UMich lecture's layer-by-layer table (single-GPU channel counts 64-192-384-256-256):

In [ ]:
alexnet = [("conv", 64, 11, 4, 2), ("pool", 3, 2, 0), ("conv", 192, 5, 1, 2), ("pool", 3, 2, 0),
           ("conv", 384, 3, 1, 1), ("conv", 256, 3, 1, 1), ("conv", 256, 3, 1, 1), ("pool", 3, 2, 0),
           ("flatten",), ("fc", 4096), ("fc", 4096), ("fc", 1000)]
rows, mem_alex, params_alex, macs_alex = analyze((3, 227, 227), alexnet, name="AlexNet, input 3 x 227 x 227")

**Where do memory, parameters, and compute live?**

In [ ]:
mem, par, fl = (np.array([r[i] for r in rows]) for i in (2, 3, 4))
conv_rows = [i for i, r in enumerate(rows) if r[0].startswith("conv")]
fc_rows = [i for i, r in enumerate(rows) if r[0].startswith("fc")]
print(f"memory in the first conv layer + pool: {mem[:2].sum() / mem.sum():.0%}")
print(f"parameters in the fc layers:          {par[fc_rows].sum() / par.sum():.0%}")
print(f"multiply-adds in the conv layers:     {fl[conv_rows].sum() / fl.sum():.0%}")

**What to notice — trends that hold for most CNNs:**
* **Most memory is in the early conv layers** (largest spatial size).
* **Nearly all parameters are in the fully connected layers.**
* **Most compute is in the conv layers.**

How were these layer sizes chosen? *Trial and error.* AlexNet's design is hand-tuned, which is exactly what VGG later replaced with simple rules.

---
## 3. ZFNet (Zeiler & Fergus, 2013): a bigger AlexNet

ILSVRC 2013 winner (16.4% → 11.7%) with essentially the same architecture **made bigger**:
* **conv1: 11×11 stride 4 → 7×7 stride 2** (less aggressive downsampling, so later layers work at higher resolution);
* **conv3, 4, 5: 512, 1024, 512 filters** instead of 384, 384, 256.

**More trial and error.** The cost:

In [ ]:
zfnet = [("conv", 64, 7, 2, 1), ("pool", 3, 2, 0), ("conv", 192, 5, 2, 0), ("pool", 3, 2, 0),
         ("conv", 512, 3, 1, 1), ("conv", 1024, 3, 1, 1), ("conv", 512, 3, 1, 1), ("pool", 3, 2, 0),
         ("flatten",), ("fc", 4096), ("fc", 4096), ("fc", 1000)]
_, mem_zf, params_zf, macs_zf = analyze((3, 224, 224), zfnet, verbose=False)
print(f"AlexNet:     {params_alex / 1e6:.1f}M params, {macs_alex / 1e9:.2f} GMACs, {mem_alex / 1024:.1f} MB activations")
print(f"ZFNet-style: {params_zf / 1e6:.1f}M params, {macs_zf / 1e9:.2f} GMACs, {mem_zf / 1024:.1f} MB activations")
print("(an approximation of ZFNet: the paper's exact layer sizes differ slightly)")

---
## 4. VGG: deeper networks, simple rules (Simonyan & Zisserman, 2015)

**VGG design rules:**
* **All convolutions are 3×3, stride 1, pad 1.**
* **All max pools are 2×2, stride 2.**
* **After each pool, double the number of channels.**

The network has **5 convolutional stages** (2, 2, 3, 3, 3 conv layers for VGG-16; 2, 2, 4, 4, 4 for VGG-19), each followed by a pool, then 3 FC layers. 2nd place in ILSVRC 2014 (7.3%). *Small filters, deeper networks* **[CS231n 2026]**. Its config is so regular it fits in one list (numbers = conv channels, "M" = max pool):

In [ ]:
def vgg(cfg):
    # Build a VGG layer list from a config of channel counts and "M" (max pool).
    layers = [("pool", 2, 2, 0) if v == "M" else ("conv", v, 3, 1, 1) for v in cfg]
    return layers + [("flatten",), ("fc", 4096), ("fc", 4096), ("fc", 1000)]

vgg16 = vgg([64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512, "M", 512, 512, 512, "M"])
vgg19 = vgg([64, 64, "M", 128, 128, "M", 256, 256, 256, 256, "M", 512, 512, 512, 512, "M", 512, 512, 512, 512, "M"])
rows16, mem16, params16, macs16 = analyze((3, 224, 224), vgg16, name="VGG-16, input 3 x 224 x 224")

**Check against the published parameter counts**, and compare with AlexNet:

In [ ]:
_, mem19, params19, macs19 = analyze((3, 224, 224), vgg19, verbose=False)
print(f"VGG-16: {params16:,} parameters (published / torchvision: 138,357,544)")
print(f"VGG-19: {params19:,} parameters (published / torchvision: 143,667,240)")
fc_params16 = sum(r[3] for r in rows16 if r[0].startswith("fc"))
print(f"VGG-16's fc layers hold {fc_params16 / params16:.0%} of its parameters")
print(f"\nAlexNet vs VGG-16: memory {mem_alex / 1024:.1f} vs {mem16 / 1024:.1f} MB ({mem16 / mem_alex:.0f}x);  "
      f"params {params_alex / 1e6:.1f}M vs {params16 / 1e6:.1f}M ({params16 / params_alex:.1f}x);  "
      f"MACs {macs_alex / 1e9:.2f}G vs {macs16 / 1e9:.2f}G ({macs16 / macs_alex:.0f}x)")

### 4.1 Why 3×3 filters?
* **Option 1:** one conv(5×5, C → C): **params $25C^2$, FLOPs $25C^2HW$**.
* **Option 2:** two conv(3×3, C → C): **params $18C^2$, FLOPs $18C^2HW$**.

**Two 3×3 convs have the same receptive field as one 5×5** (Lecture 7: 1 + 2·2 = 5), **but fewer parameters and less computation**, and an extra nonlinearity between them. **[CS231n 2026]** makes the same argument with three layers: **three 3×3 convs = one 7×7** in receptive field, with $3\cdot9C^2 = 27C^2$ parameters instead of $49C^2$. Computed (in units of $C^2$, no biases):

In [ ]:
C_, Hs = 256, 28
for desc, n_layers, K_ in [("one 5x5", 1, 5), ("two 3x3", 2, 3), ("one 7x7", 1, 7), ("three 3x3", 3, 3)]:
    layers = [("conv", C_, K_, 1, K_ // 2, 1, False)] * n_layers
    _, _, p_, f_ = analyze((C_, Hs, Hs), layers, verbose=False)
    print(f"{desc:10s}: params {p_ / C_ ** 2:3.0f} C^2,  MACs {f_ / (C_ ** 2 * Hs * Hs):3.0f} C^2 HW")

### 4.2 Why double the channels after each pool?
Compare one conv in stage $i$ (input $C\times2H\times2W$) with one in stage $i + 1$ (input $2C\times H\times W$):
* Stage $i$, conv(3×3, C → C): memory $4HWC$, params $9C^2$, FLOPs $36HWC^2$.
* Stage $i + 1$, conv(3×3, 2C → 2C): memory $2HWC$, params $36C^2$, **FLOPs $36HWC^2$: the same.**

So **every stage costs the same compute per conv layer**:

In [ ]:
for i, (size, ch) in enumerate([(224, 64), (112, 128), (56, 256), (28, 512)]):
    _, _, _, f_ = analyze((ch, size, size), [("conv", ch, 3, 1, 1)], verbose=False)
    print(f"stage {i + 1}: 3x3 conv {ch:3d} -> {ch:3d} at {size:3d} x {size:3d}: {f_ / 1e9:.2f} GMACs")

---
## 5. GoogLeNet: focus on efficiency (Szegedy et al., 2015)

ILSVRC 2014 winner (6.7%), **22 layers**, and **many innovations for efficiency** (fewer parameters, less memory, less computation).

### 5.1 The stem network
**Aggressively downsample the input at the very start** (in VGG-16, most compute was at the high-resolution start): conv 7×7 stride 2, max pool, 1×1 conv, 3×3 conv, max pool takes 224×224 → 28×28. Compared with VGG-16's layers down to the same resolution:

In [ ]:
googlenet_stem = [("conv", 64, 7, 2, 3), ("pool", 3, 2, 1), ("conv", 64, 1, 1, 0), ("conv", 192, 3, 1, 1), ("pool", 3, 2, 1)]
_, mem_g, params_g, macs_g = analyze((3, 224, 224), googlenet_stem, name="GoogLeNet stem: 224 -> 28")
_, mem_v, params_v, macs_v = analyze((3, 224, 224), vgg16[:10], verbose=False)       # VGG-16's first 3 stages also end at 28x28
print(f"\nVGG-16 up to 28 x 28: memory {mem_v / 1024:.1f} MB ({mem_v / mem_g:.1f}x), params {params_v / 1e6:.2f}M ({params_v / params_g:.1f}x), "
      f"MACs {macs_v / 1e9:.2f}G ({macs_v / macs_g:.1f}x)")

### 5.2 The Inception module
A **local unit with parallel branches**, repeated throughout the network: **1×1, 3×3, and 5×5 convolutions and a 3×3 max pool**, applied to the same input and **concatenated** along the channel axis. It tries several kernel sizes at once instead of making kernel size a hyperparameter. To keep it cheap, it uses **1×1 "bottleneck" convolutions to reduce the number of channels before the expensive 3×3 and 5×5 convolutions** (Lecture 7: a 1×1 conv is a per-pixel FC layer).

**The first module ("3a")**, input 192×28×28. `branch_cost` adds up a chain of convolutions:

In [ ]:
def branch_cost(C_in, H_, convs):
    # A chain of 'same'-padded convolutions [(C_out, K), ...] on a C_in x H_ x H_ input. Returns (params, MACs, C_out).
    params = macs = 0; C = C_in
    for C_out, K_ in convs:
        params += C_out * C * K_ * K_ + C_out; macs += C_out * H_ * H_ * C * K_ * K_; C = C_out
    return params, macs, C

C_in, H_ = 192, 28
naive = [branch_cost(C_in, H_, [(64, 1)]), branch_cost(C_in, H_, [(128, 3)]), branch_cost(C_in, H_, [(32, 5)])]
naive_out = 64 + 128 + 32 + C_in                                       # the pool branch passes all 192 channels through
with_bottlenecks = [branch_cost(C_in, H_, [(64, 1)]),
                    branch_cost(C_in, H_, [(96, 1), (128, 3)]),        # 1x1 down to 96, then 3x3
                    branch_cost(C_in, H_, [(16, 1), (32, 5)]),         # 1x1 down to 16, then 5x5
                    branch_cost(C_in, H_, [(32, 1)])]                  # pool, then a 1x1 projection
bottleneck_out = 64 + 128 + 32 + 32
for name, branches, out_c in [("naive (no 1x1 reductions)", naive, naive_out), ("with 1x1 bottlenecks", with_bottlenecks, bottleneck_out)]:
    print(f"{name:28s}: params {sum(b[0] for b in branches) / 1e3:6.1f}k, MACs {sum(b[1] for b in branches) / 1e6:6.1f}M, "
          f"output channels {out_c}")

**What to notice:** the bottlenecks cut the module's compute by more than half and stop the channels from piling up (in the naive version, the pool branch alone carries all 192 input channels forward, at every module).

### 5.3 Global average pooling instead of big FC layers
**No large FC layers at the end.** **Global average pooling** averages each channel's spatial map to one number (1024×7×7 → 1024), then **one linear layer** gives the class scores:

In [ ]:
gap_head = 1024 * 1000 + 1000
flatten_head = 1024 * 7 * 7 * 1000 + 1000
print(f"global avg pool -> fc 1000: {gap_head:,} params")
print(f"flatten 1024x7x7 -> fc 1000: {flatten_head:,} params ({flatten_head / gap_head:.0f}x)")
print(f"(VGG-16's three fc layers: {fc_params16:,} params)")

### 5.4 Auxiliary classifiers
**Training with the loss only at the end didn't work well: the network is too deep, and gradients don't propagate cleanly.** As a hack, GoogLeNet attached **auxiliary classifiers** at intermediate points, which also try to classify the image and receive their own loss. **GoogLeNet came before batch normalization; with BN, we no longer need this trick.**

---
## 6. ResNet: residual networks (He et al., 2016)

**Once we have batch normalization, we can train networks with 10+ layers. What happens as we go deeper?**

**A deeper model does worse than a shallower one!** The 56-layer network has higher **training** error *and* test error than a 20-layer one (CIFAR-10).
* *Initial guess:* the deep model is **overfitting** (it's bigger).
* *In fact,* it's **underfitting**: it does worse **on the training set** too, so it's not overfitting.

**A deeper model can always emulate a shallower one**: copy the shallow model's layers and set the extra layers to the **identity**. So deeper should be at least as good. **Hypothesis: this is an optimization problem. Deep models are harder to optimize, and in particular don't easily learn identity functions.** **Solution: change the network so learning identity functions with the extra layers is easy!**

**The residual block:** instead of making the layers fit the desired mapping $H(x)$ directly ("plain" layers), have them fit the **residual** $F(x) = H(x) - x$, and add a **shortcut**:
$$\text{output} = F(x) + x$$
**If all the block's weights are zero, it computes the identity.**

**Let's reproduce both the problem and the fix** with deep fully connected networks on the digits (He initialization, no batch norm; same width, optimizer, and number of steps). **Step 1: data and loss.**

In [ ]:
digits = load_digits(); Xd = digits.data / 16.0 - 0.3; yd = digits.target

def softmax_xent(S_, y_):
    # Softmax loss and its gradient with respect to the scores.
    S_ = S_ - S_.max(1, keepdims=True); P_ = np.exp(S_); P_ /= P_.sum(1, keepdims=True)
    loss = -np.log(P_[np.arange(len(y_)), y_] + 1e-12).mean(); d = P_.copy(); d[np.arange(len(y_)), y_] -= 1
    return loss, d / len(y_)

**Step 2: the model.** An input layer, then `n_blocks` blocks of two linear layers, then an output layer. Each block computes either
* **plain:** $h\leftarrow\text{ReLU}(W_2\,\text{ReLU}(W_1h))$, or
* **residual:** $h\leftarrow\text{ReLU}(h + W_2\,\text{ReLU}(W_1h))$.

In the backward pass, the shortcut adds the upstream gradient **directly** to the block's input gradient (Lecture 6's add node: a gradient distributor). The residual branches start with small weights (0.1× the usual scale), so each block starts close to the identity, a common practical trick.

In [ ]:
class DeepNet:
    # 64 -> width -> [n_blocks blocks of 2 linear layers] -> 10, plain or residual.
    def __init__(self, n_blocks, width=32, residual=False, seed=0):
        r = np.random.default_rng(seed); self.residual = residual
        he = lambda fan_in, shape: r.normal(0, math.sqrt(2 / fan_in), shape)
        self.W_in = he(64, (64, width))
        self.W1 = [he(width, (width, width)) for _ in range(n_blocks)]
        self.W2 = [he(width, (width, width)) * (0.1 if residual else 1.0) for _ in range(n_blocks)]
        self.W_out = he(width, (width, 10))

    def step(self, X, y, lr):
        # Forward, backward, and an SGD update. Returns the loss.
        h = np.maximum(X @ self.W_in, 0); cache = []
        for W1, W2 in zip(self.W1, self.W2):                         # forward through the blocks
            z1 = h @ W1; a1 = np.maximum(z1, 0); z2 = a1 @ W2
            pre = (h + z2) if self.residual else z2                  # the shortcut adds the block's input
            cache.append((h, z1, a1, pre)); h = np.maximum(pre, 0)
        loss, dS = softmax_xent(h @ self.W_out, y)
        gW_out = h.T @ dS; dh = dS @ self.W_out.T
        for i in reversed(range(len(self.W1))):                      # backward through the blocks
            h_in, z1, a1, pre = cache[i]
            dpre = dh * (pre > 0)                                    # through the final ReLU
            gW2 = a1.T @ dpre
            dz1 = (dpre @ self.W2[i].T) * (z1 > 0)
            gW1 = h_in.T @ dz1
            dh = dz1 @ self.W1[i].T + (dpre if self.residual else 0) # shortcut: the gradient also flows straight through
            self.W1[i] -= lr * gW1; self.W2[i] -= lr * gW2
        dz0 = dh * (X @ self.W_in > 0)
        self.W_in -= lr * (X.T @ dz0); self.W_out -= lr * gW_out
        return loss

**Step 3: train shallow and deep plain networks, and a deep residual one** (3 random seeds each; we report the training loss averaged over the last 100 steps):

In [ ]:
def final_training_loss(n_blocks, residual, steps=1500, lr=0.02, seed=0):
    net = DeepNet(n_blocks, residual=residual, seed=seed); r = np.random.default_rng(seed); history = []
    for t in range(steps):
        idx = r.integers(0, 1300, 64)
        with np.errstate(all="ignore"):
            history.append(net.step(Xd[idx], yd[idx], lr))
    return np.mean(history[-100:])

t0 = time.time()
print(f"{'network':22s} {'layers':>6s} {'final training loss':>20s}")
for n_blocks, residual in [(2, False), (10, False), (25, False), (25, True)]:
    results = [final_training_loss(n_blocks, residual, seed=s) for s in range(3)]
    name = ("residual" if residual else "plain") + f", {n_blocks} blocks"
    print(f"{name:22s} {2 * n_blocks + 2:6d} {np.mean(results):20.4f}   (seeds: {np.round(results, 3)})")
print(f"({time.time() - t0:.0f}s)")

**What to notice:** the ResNet paper's figure in miniature. The 52-layer **plain** network can't even fit its **training** data (its loss stays near the chance level of 2.30), while the shallow plain network fits it easily. Adding shortcuts to **the very same 52-layer network** fixes it completely. **Why, in gradient terms:** the shortcut's local gradient is the identity, so $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial\text{out}}\left(I + \frac{\partial F}{\partial x}\right)$: the gradient reaches early layers through the shortcuts without being multiplied by every layer's weights.

### 6.1 The full ResNet architecture
* **A stack of many residual blocks.** **Regular design, like VGG:** each block has **two 3×3 conv layers**.
* **Divided into stages:** the first block of each stage **halves the resolution** (stride-2 conv) and **doubles the channels**.
* **The same aggressive stem as GoogLeNet** (7×7 conv stride 2, max pool) downsamples 4× before the blocks.
* **Like GoogLeNet, no big FC layers:** **global average pooling** and a **single linear layer**.
* **Total depths of 18, 34, 50, 101, or 152 layers.** ResNet-152 won ILSVRC 2015 (3.57% top-5) and **swept all classification and detection competitions in ILSVRC and COCO 2015!**

**Counting every parameter.** torchvision's ResNets put **batch norm after every convolution** (so convs have no bias; each BN has $2C$ parameters, γ and β), and when a block changes shape, its shortcut is a **1×1 conv + BN "projection"**. `resnet_cost` walks through the whole network with these rules:

In [ ]:
def resnet_cost(block, stage_blocks, H=224):
    # Exact parameter count and multiply-adds of a torchvision-style ResNet.
    # block: "basic" (two 3x3) or "bottleneck" (1x1, 3x3, 1x1 with 4x expansion); stage_blocks: blocks per stage.
    total = {"params": 0, "macs": 0}
    def conv_bn(C_in, C_out, K_, S_, H_in):
        # A conv (no bias) followed by BN. Adds to the totals; returns the output size.
        H_out = (H_in + 2 * (K_ // 2) - K_) // S_ + 1
        total["params"] += C_out * C_in * K_ * K_ + 2 * C_out
        total["macs"] += C_out * H_out * H_out * C_in * K_ * K_
        return H_out

    size = conv_bn(3, 64, 7, 2, H)                     # stem: 7x7 / 2 conv + BN ...
    size = (size + 2 - 3) // 2 + 1                     # ... then 3x3 / 2 max pool  -> 56 x 56
    C_in, expansion = 64, (1 if block == "basic" else 4)
    for stage, (n, width) in enumerate(zip(stage_blocks, [64, 128, 256, 512])):
        for b in range(n):
            S_ = 2 if (b == 0 and stage > 0) else 1    # first block of stages 2-4 halves the resolution
            C_out = width * expansion
            if block == "basic":
                s1 = conv_bn(C_in, width, 3, S_, size); s2 = conv_bn(width, width, 3, 1, s1)
            else:                                      # bottleneck, stride in the 3x3 (torchvision's "v1.5")
                s1 = conv_bn(C_in, width, 1, 1, size); s2 = conv_bn(width, width, 3, S_, s1); s2 = conv_bn(width, C_out, 1, 1, s2)
            if S_ != 1 or C_in != C_out:
                conv_bn(C_in, C_out, 1, S_, size)      # projection shortcut
            C_in, size = C_out, s2
    total["params"] += C_in * 1000 + 1000              # global average pool + fc
    total["macs"] += C_in * 1000
    return total["params"], total["macs"]

**Compare with torchvision's exact parameter counts:**

In [ ]:
published = {"ResNet-18": 11_689_512, "ResNet-34": 21_797_672, "ResNet-50": 25_557_032, "ResNet-101": 44_549_160, "ResNet-152": 60_192_808}
configs = {"ResNet-18": ("basic", [2, 2, 2, 2]), "ResNet-34": ("basic", [3, 4, 6, 3]), "ResNet-50": ("bottleneck", [3, 4, 6, 3]),
           "ResNet-101": ("bottleneck", [3, 4, 23, 3]), "ResNet-152": ("bottleneck", [3, 8, 36, 3])}
resnet_numbers = {}
print(f"{'model':11s} {'computed params':>16s} {'torchvision':>13s} {'match':>6s} {'GMACs':>7s}")
for name, (blk, n) in configs.items():
    p_, f_ = resnet_cost(blk, n); resnet_numbers[name] = (p_, f_)
    print(f"{name:11s} {p_:16,d} {published[name]:13,d} {str(p_ == published[name]):>6s} {f_ / 1e9:7.2f}")
print(f"(VGG-16 for comparison: {params16 / 1e6:.0f}M params, {macs16 / 1e9:.1f} GMACs)")

**What to notice:** **exact** matches for all five. The compute also matches the commonly cited numbers (torchvision lists 1.81, 3.66, 4.09, 7.80, 11.51 "GFLOPs", which are multiply-adds; the ResNet paper's 3.8, 7.6, 11.3 for the deeper models put the stride in the first 1×1 conv instead, which is a little cheaper). **ResNet-152 is 152 layers deep and still cheaper than VGG-16.**

### 6.2 Bottleneck blocks
For deeper networks, ResNet uses a **bottleneck block**: 1×1 conv reducing $4C\to C$, 3×3 conv $C\to C$, 1×1 conv expanding $C\to4C$. Compare with a **basic block** (two 3×3 convs on $C$ channels):
* **Basic:** $9HWC^2 + 9HWC^2 = 18HWC^2$.
* **Bottleneck:** $4HWC^2 + 9HWC^2 + 4HWC^2 = 17HWC^2$.

**More layers, less computation!** (And 4× more channels at the block's input and output.) **ResNet-50 = ResNet-34 with bottleneck blocks** at nearly the same cost, and **deeper is better**: error falls steadily from ResNet-18 to ResNet-152. Computed:

In [ ]:
C_, Hs = 64, 56
_, _, _, f_basic = analyze((C_, Hs, Hs), [("conv", C_, 3, 1, 1, 1, False)] * 2, verbose=False)
_, _, _, f_bottle = analyze((4 * C_, Hs, Hs), [("conv", C_, 1, 1, 0, 1, False), ("conv", C_, 3, 1, 1, 1, False),
                                               ("conv", 4 * C_, 1, 1, 0, 1, False)], verbose=False)
print(f"basic block: {f_basic / (Hs * Hs * C_ ** 2):.0f} HWC^2;  bottleneck block: {f_bottle / (Hs * Hs * C_ ** 2):.0f} HWC^2")

---
## 7. Improving residual networks

### 7.1 Block design: pre-activation (He et al., 2016)
In the original block, the ReLU comes **after** the addition: $\text{out} = \text{ReLU}(x + F(x))$. **That ReLU means the block can't actually learn the identity** for negative inputs, even with $F = 0$, since $\text{ReLU}(x)\neq x$ when $x < 0$. The **pre-activation** block reorders to BN → ReLU → conv → BN → ReLU → conv, so the shortcut is a clean identity from block to block. **Slight improvement** (ImageNet top-1 error: ResNet-152 21.3% → 21.1%; ResNet-200 21.8% → 20.7%), but **not used that much in practice**.

In [ ]:
x_in = np.random.default_rng(0).normal(size=8)
F_zero = np.zeros(8)                                           # a residual branch whose weights are all zero
print("original block,       ReLU(x + F(x)) with F = 0 equals x?", np.allclose(np.maximum(x_in + F_zero, 0), x_in), "(negative entries get clipped)")
print("pre-activation block, x + F(x)       with F = 0 equals x?", np.allclose(x_in + F_zero, x_in))

### 7.2 ResNeXt: grouped convolutions (Xie et al., 2017)
Make the bottleneck block **wider with $G$ parallel paths**, each 1×1 conv $4C\to c$, 3×3 conv $c\to c$, 1×1 conv $c\to4C$, summed together. That's equivalent to a **grouped convolution** (Lecture 7: split the channels into $G$ groups; each group's filters only see their group; `groups=G` in PyTorch).

**Equal-cost condition.** One path costs $(4Cc + 9c^2 + 4Cc)HW = (8Cc + 9c^2)HW$, so $G$ paths match the bottleneck's $17C^2HW$ when
$$9Gc^2 + 8GCc - 17C^2 = 0\qquad\Longrightarrow\qquad c = \frac{-8GC + \sqrt{(8GC)^2 + 4\cdot9G\cdot17C^2}}{2\cdot9G}$$
(the positive root of the quadratic in $c$). **Adding groups improves performance at the same computational cost!** (ResNeXt-50 32×4d beats ResNet-50 at equal FLOPs.)

In [ ]:
C_ = 64
print("equal-cost path width c for C = 64:")
for G in [1, 2, 4, 8, 32]:
    c = (-8 * G * C_ + math.sqrt((8 * G * C_) ** 2 + 4 * 9 * G * 17 * C_ ** 2)) / (2 * 9 * G)
    print(f"   G = {G:2d} groups -> c = {c:5.1f} channels per path")
print("(UMich's examples: G = 4 -> c = 24; G = 32 -> c = 4; G = 1 is the plain bottleneck, c = C)")

### 7.3 Ensembles, and squeeze-and-excitation
* **ILSVRC 2016:** a **multi-scale ensemble** of Inception, Inception-ResNet, ResNet, and Wide ResNet models (Trimps-Soushen): 2.99%.
* **ILSVRC 2017: Squeeze-and-Excitation Networks** (SENet; Hu et al. 2018): add a small **channel-attention** branch to each block: average-pool each channel to one number ("squeeze"), pass the $C$-vector through FC → ReLU → FC → sigmoid ("excitation"), and multiply each channel by its weight. Cheap and effective: 2.25%. **The challenge was then completed** and moved to Kaggle.

---
## 8. Efficient networks, and searching for architectures

**After ResNet, efficiency became central:** *how can we improve accuracy without increasing complexity?* Lots of tiny networks for mobile devices:
* **MobileNets** (Howard et al. 2017): replace each standard convolution with a **depthwise-separable** one: a **depthwise** 3×3 conv (groups = channels: each channel filtered separately) followed by a **pointwise** 1×1 conv that mixes channels. Cost per pixel: $9C + C^2$ instead of $9C^2$, a saving of $\frac{9C^2}{9C + C^2} = \frac{1}{1/C + 1/9}$, close to 9× for large $C$.
* **ShuffleNet** (Zhang et al. 2018): grouped 1×1 convs plus a **channel shuffle**, so information mixes between groups.

Computed, along with the cost of an SE block:

In [ ]:
for C_ in [64, 256, 1024]:
    standard, separable = 9 * C_ * C_, 9 * C_ + C_ * C_
    print(f"C = {C_:4d}: standard 3x3 {standard:>10,d} MACs/pixel, depthwise-separable {separable:>9,d}  "
          f"({standard / separable:.1f}x cheaper; formula {1 / (1 / C_ + 1 / 9):.1f}x)")
C_, reduction = 256, 16
print(f"SE block for C = {C_} (reduction 16): {2 * C_ * C_ // reduction:,} extra parameters, vs. {9 * C_ * C_:,} for one 3x3 conv")

**Neural Architecture Search (NAS)** (Zoph & Le, 2017): *designing architectures is hard, so let's automate it!* A **controller** network (an RNN) outputs architectures; train the sampled **child networks**; then update the controller with policy gradient (Lecture 21) to make good architectures more likely. **Very expensive!** The original search trained over ten thousand child networks on hundreds of GPUs for weeks. Later work made it far cheaper, and searched architectures (NASNet, AmoebaNet) reached better accuracy per FLOP than hand-designed ones.

🔎 **After 2019** (beyond the UMich lecture):
* **EfficientNet** (Tan & Le 2019): **compound scaling**: grow depth, width, and input resolution together with fixed ratios, from a NAS-found base network.
* **RegNet** (Radosavovic et al. 2020): search over whole **design spaces**, finding simple rules (widths that grow linearly with depth).
* **Vision Transformers** (Dosovitskiy et al. 2021; Lecture 13): an image as a sequence of 16×16 patches, fed to a Transformer; with enough data they match or beat CNNs.
* **ConvNeXt** (Liu et al. 2022): "modernize" a ResNet step by step (7×7 depthwise kernels, LayerNorm, GELU, fewer activations, inverted bottlenecks) and it matches Swin Transformers: the training recipe and scale matter as much as the architecture family.

---
## 9. Comparing complexity, and which architecture to use

Canziani et al. (2017) plotted ImageNet accuracy against operations, with circle size = parameters. Using torchvision's reported top-1 accuracies (original `IMAGENET1K_V1` weights) and our computed costs:

In [ ]:
zoo = {  # name: (top-1 %, GMACs, params in millions); compute and params from this notebook where we computed them
    "AlexNet": (56.5, macs_alex / 1e9, params_alex / 1e6), "VGG-16": (71.6, macs16 / 1e9, params16 / 1e6), "GoogLeNet": (69.8, 1.50, 6.6),
    "ResNet-18": (69.8, resnet_numbers["ResNet-18"][1] / 1e9, resnet_numbers["ResNet-18"][0] / 1e6),
    "ResNet-34": (73.3, resnet_numbers["ResNet-34"][1] / 1e9, resnet_numbers["ResNet-34"][0] / 1e6),
    "ResNet-50": (76.1, resnet_numbers["ResNet-50"][1] / 1e9, resnet_numbers["ResNet-50"][0] / 1e6),
    "ResNet-101": (77.4, resnet_numbers["ResNet-101"][1] / 1e9, resnet_numbers["ResNet-101"][0] / 1e6),
    "ResNet-152": (78.3, resnet_numbers["ResNet-152"][1] / 1e9, resnet_numbers["ResNet-152"][0] / 1e6),
    "MobileNetV2": (71.9, 0.30, 3.5),
}
plt.figure(figsize=(8, 5))
for name, (acc_, g_, p_) in zoo.items():
    plt.scatter(g_, acc_, s=p_ * 4, alpha=0.5); plt.annotate(name, (g_, acc_), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.xscale("log"); plt.xlabel("GMACs per image (log scale)"); plt.ylabel("ImageNet top-1 accuracy (%)")
plt.title("accuracy vs. compute (circle area ~ parameters)"); plt.show()
for name, (acc_, g_, p_) in zoo.items():
    print(f"{name:12s} top-1 {acc_:5.1f}%   {g_:6.2f} GMACs   {p_:6.1f}M params")

**Reading the chart:** **VGG** has the most memory and operations; **GoogLeNet** is very efficient; **AlexNet** has little compute but lots of parameters (and low accuracy); **ResNet**: simple design, moderate efficiency, high accuracy; **MobileNet**-style networks reach VGG-level accuracy at a tiny fraction of the compute.

**The story in one paragraph:** early work (AlexNet → ZFNet → VGG) showed **bigger networks work better**; GoogLeNet was one of the first to **focus on efficiency** (aggressive stem, 1×1 bottlenecks, global average pooling); **ResNet showed how to train extremely deep networks**, limited mainly by GPU memory, and showed **diminishing returns** as networks grew. After ResNet, **efficient networks** took center stage (MobileNet, ShuffleNet), and **NAS** began automating design.

### Which architecture should I use?
**Don't be a hero.** For most problems, use an **off-the-shelf architecture**; don't design your own! **If you just care about accuracy, ResNet-50 or ResNet-101 are great choices.** **If you need efficiency** (real-time, mobile), **try MobileNets and ShuffleNets.** 🔎 In 2026: start from a **pretrained** model in a model zoo (torchvision, `timm`, Hugging Face: ResNet, ConvNeXt, EfficientNet, ViT) and **fine-tune** it (transfer learning, Lecture 11), rather than training from scratch.

---
## 10. PyTorch: the model zoo (run in Colab)

In [ ]:
# [PyTorch] Needs: pip install torch torchvision. weights=None builds the architecture without downloading weights.
import torch, torchvision
for name in ["alexnet", "vgg16", "googlenet", "resnet18", "resnet50", "resnet152", "mobilenet_v2"]:
    kwargs = {"aux_logits": False, "init_weights": False} if name == "googlenet" else {}
    model = getattr(torchvision.models, name)(weights=None, **kwargs)
    print(f"{name:13s}: {sum(p.numel() for p in model.parameters()):>12,d} parameters")   # compare with Secs. 4 and 6
# Transfer learning in three lines: load pretrained weights, replace the classifier, fine-tune.
model = torchvision.models.resnet50(weights="IMAGENET1K_V1")
model.fc = torch.nn.Linear(model.fc.in_features, 10)      # a new head for 10 classes
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

---
## 11. Summary

| Architecture | Key ideas | Cost (computed here) |
|---|---|---|
| AlexNet (2012) | 5 conv + 3 FC, ReLU, 2 GPUs | memory in early convs, parameters in FC layers, compute in convs |
| ZFNet (2013) | A bigger AlexNet (7×7/2 stem, wider convs) | more trial and error |
| VGG (2014) | Only 3×3 convs and 2×2 pools; double channels after each pool | 138M params, ~15.5 GMACs; equal compute per stage |
| GoogLeNet (2014) | Aggressive stem, Inception modules with 1×1 bottlenecks, global average pooling, auxiliary classifiers | far cheaper than VGG |
| ResNet (2015) | Residual blocks $F(x) + x$ fix the degradation problem; stages; bottleneck blocks | 18 → 152 layers; ResNet-152 cheaper than VGG-16 |
| Improvements | Pre-activation, ResNeXt (groups), SENet, ensembles | same FLOPs, better accuracy |
| Efficient nets | MobileNet (depthwise-separable), ShuffleNet, NAS; 🔎 EfficientNet, RegNet, ConvNeXt, ViT | accuracy per FLOP |

### Check yourself
<details><summary>1. Where are most of AlexNet's parameters, memory, and FLOPs?</summary>

Parameters: fully connected layers (96% in §2). Memory: early conv layers (41% in the first conv + pool). FLOPs: conv layers (92%).
</details>

<details><summary>2. Why does VGG use stacks of 3×3 convolutions?</summary>

Two (three) 3×3 layers have the receptive field of one 5×5 (7×7), with fewer parameters and FLOPs (18C² vs 25C²; 27C² vs 49C²) and more nonlinearities.
</details>

<details><summary>3. What problem did ResNet solve, and why isn't it overfitting?</summary>

Very deep plain networks had higher **training** error than shallower ones: an optimization problem (they can't easily learn identity mappings), not overfitting, which would show low training error. Residual connections make the identity the default (F = 0), so extra layers can't make things worse.
</details>

<details><summary>4. Why is a bottleneck block cheaper than a basic block?</summary>

The expensive 3×3 conv runs on C channels between two 1×1 convs that reduce from and expand back to 4C: 17HWC² vs 18HWC², with more layers and 4× wider input/output.
</details>

<details><summary>5. How does global average pooling reduce parameters?</summary>

It averages each channel to one number (1024×7×7 → 1024) before the final linear layer, which then needs 1024×1000 weights instead of 50,176×1000.
</details>

### Exercises
1. Add **ResNeXt-50 (32×4d)** to `resnet_cost` (bottleneck width doubled, grouped 3×3 conv with 32 groups) and check its count against torchvision's 25,028,904.
2. Use `analyze` to cost a **MobileNetV1** (depthwise 3×3 + pointwise 1×1 blocks) and compare its MACs with VGG-16's.
3. In §6, increase the depth to 40 and 60 blocks. Does the residual network keep training? What happens without the 0.1 scaling of the residual branches?
4. Add batch normalization to `DeepNet` (reuse Lecture 7's `BatchNorm`) and test whether the degradation problem remains for plain networks **with** BN, as in the ResNet paper.
5. Compute VGG-16's and ResNet-50's activation memory for a batch of 64 during training (sum the outputs of every conv layer).

### References
* Krizhevsky, Sutskever & Hinton 2012; Zeiler & Fergus 2014; Simonyan & Zisserman 2015; Szegedy et al. 2015.
* He et al. 2016 (*Deep Residual Learning*; *Identity Mappings*); Xie et al. 2017 (ResNeXt); Hu, Shen & Sun 2018 (SENet).
* Howard et al. 2017 (MobileNets); Sandler et al. 2018 (MobileNetV2); Zhang et al. 2018 (ShuffleNet); Zoph & Le 2017 (NAS).
* Tan & Le 2019 (EfficientNet); Radosavovic et al. 2020 (RegNet); Dosovitskiy et al. 2021 (ViT); Liu et al. 2022 (ConvNeXt).
* Canziani, Paszke & Culurciello 2017.

**Next:** Lecture 9, Hardware and Software.